## Configuracion y Entorno

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Cargar los datos desde la carpeta de procesados
df = pd.read_csv('customer-segmentation-analysis\data\processed\df_clean.csv')
df_scaled = pd.read_csv('customer-segmentation-analysis\data\processed\df_scaled.csv')


## Desarrollo del Modelo


In [ ]:
# 1. Configuración del Modelo
# Usamos init='k-means++' para que la ubicación inicial de los centros sea inteligente y converja más rápido.
# Fijamos un random_state para que los resultados sean exactamente los mismos.
# El número de clusters (n_clusters) lo definimos en 3, por la interpretacion que tuvimos del dataset.
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42)

# 2. Entrenamiento del Modelo
kmeans.fit(df_scaled)

# 3. Asignación de Etiquetas
# Extraemos el número de grupo (0, 1 o 2) y se lo pegamos a nuestra tabla ORIGINAL (no a la estandarizada).
# Esto con el objetivo de poder interpreatr los resultados de manera más sencilla.
df['Cluster'] = kmeans.labels_

# Vista rápida de cuántos clientes hay en cada segmento
#print("Distribución de Clientes por Segmento:")
print(df['Cluster'].value_counts())

In [ ]:
# 4. Interpretación Financiera de los Segmentos
perfil_clusters = df.groupby('Cluster').agg({
    'Total_Spent': 'mean',        
    'Income': 'mean',             
    'Age': 'mean',               
    'Total_Children': 'mean',     
    'NumDealsPurchases': 'mean'   
}).round(2)

#print("\n--- Perfil Financiero y Demográfico de cada Clúster ---")
print(perfil_clusters)

## Analisis de Componentes
Ahora lo que buscamos con el PCA es visualizar nuestros clusteres y evaluar la calidad del algoritmo y simplificar la interpretacion.

In [ ]:
# 1. Configurar PCA para reducir todo a solo 2 dimensiones
pca = PCA(n_components=2)

# 2. Aplicar la transformación a nuestra matriz estandarizada
componentes_principales = pca.fit_transform(df_scaled)

# 3. Guardar los resultados en una nueva tabla temporal para graficar
df_pca = pd.DataFrame(data=componentes_principales, columns=['PC1', 'PC2'])

# 4. Pegarle la etiqueta de los clústeres que generó K-Means
df_pca['Cluster'] = df['Cluster']


In [ ]:
plt.figure(figsize=(10, 7))

# Crear el gráfico de dispersión (scatter plot)
# Usamos PC1 en el eje X, PC2 en el eje Y, y el 'hue' pinta los puntos según su clúster
sns.scatterplot(x='PC1', y='PC2', hue='Cluster', data=df_pca, palette='viridis', s=60, alpha=0.8)

plt.title('Visualización de Segmentos de Clientes usando PCA (k=3)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')

# Formatear la leyenda para que se vea profesional
plt.legend(title='Segmento')
plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

## Investigacion e Interpretacion
Se observa que en cuanto a el cluster 1 y 2 hay una diferencia clara, el algortimo funciono correctamente, pero en cuanto a el grupo 0, es visiblemente un problema, esto debido a que algunos valores atipicos pasaron el proceso de limpieza.

Lo que se hara es hacer una investigacion de estos valores para determinar que hace que este grupo este aislado, ya que pdoria presentar un grupo de personas que invierte mucho en la tienda.


In [ ]:
# Agrupamos por clúster y calculamos promedios y conteos
analisis_vip = df.groupby('Cluster').agg({
    'Total_Spent': ['mean', 'count'], 
    'Income': 'mean',                  
    'NumCatalogPurchases': 'mean',     
    'NumWebPurchases': 'mean',         
    'NumStorePurchases': 'mean',     
    'Age': 'mean'                     
}).round(1) # Redondeamos a 1 decimal para leerlo más fácil

#print("--- Análisis de Perfiles de Clientes ---")
print(analisis_vip)

Encontramos que el cluster 0 con 4 personas no aporta un valor real a nuestro algoritmo ni un valor predictivo o comercial. 
La solucion es purgar el ruido que generan estos 4 clientes y eliminarlos del dataset.

In [ ]:

# 1. Identificar el ruido (los 4 clientes) 
clientes_ruido = df[df['Cluster'] == 0].index

# 2. Eliminarlos de nuestra tabla original
df = df.drop(clientes_ruido)

# 3. Borrar la columna 'Cluster' vieja para empezar limpios
df = df.drop(columns=['Cluster'])

# 4. Volver a preparar la matriz matemática 
df_encoded = pd.get_dummies(df, drop_first=True)
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_encoded), columns=df_encoded.columns)

# 5. Volver a entrenar el modelo con los datos purgados
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42)
df['Cluster'] = kmeans.fit_predict(df_scaled)

# 6. Calcular los nuevos promedios
nuevos_perfiles = df.groupby('Cluster').agg({
    'Total_Spent': ['mean', 'count'],
    'Income': 'mean',
    'NumDealsPurchases': 'mean'
}).round(1)

#print("--- Nuevos Perfiles (Ruido Eliminado) ---")
print(nuevos_perfiles)

### El nuevo Problema y la solucion
Podemos observar que lo anteriormente realizado no dio resultados e incluso nos da resultados desastrosos y peores que los primeros, ya que tenemos un cluster masivo y dos aislados, por ahora tomaremos un enfoque distinto centrado en la optimizacion.

Aplicaremos un enfoque llamado RFM para darle al modelo restricciones en base al comportamiento comercial de los clientes, forzandolo a tomar exclusivamente variables continuas de impacto financiero (Ingresos, Gasto total, edad, hijos y compras con descuento). De esta manera quitaremos el ruido de algunos datos que quiza no representen un valor real en el agrupamiento de los clientes.

In [ ]:
# 1. Feature Selection: Nos quedamos solo con las variables de impacto comercial
columnas_clave = ['Total_Spent', 'Income', 'Age', 'Total_Children', 'NumDealsPurchases']
df_core = df[columnas_clave]

# 2. Estandarizamos exclusivamente esta matriz reducida
scaler = StandardScaler()
df_core_scaled = pd.DataFrame(scaler.fit_transform(df_core), columns=df_core.columns)

# 3. Re-entrenamos el modelo asegurando la estructura de k=3
kmeans_opt = KMeans(n_clusters=3, init='k-means++', random_state=42)
df['Cluster'] = kmeans_opt.fit_predict(df_core_scaled)

# 4. Evaluamos la nueva distribución
perfiles_optimizados = df.groupby('Cluster').agg({
    'Total_Spent': ['mean', 'count'],
    'Income': 'mean',
    'Age': 'mean'
}).round(1)

#print("--- Perfiles Optimizados (Enfoque Financiero) ---")
print(perfiles_optimizados)

### Grafica de Perfil Financiero

In [ ]:
# Configurar el lienzo
plt.figure(figsize=(14, 6))

# Gráfica 1: Gasto Total por Clúster
plt.subplot(1, 2, 1)
sns.boxplot(x='Cluster', y='Total_Spent', data=df, palette='viridis')
plt.title('Gasto Total por Segmento')
plt.xlabel('Segmento (Clúster)')
plt.ylabel('Dólares Gastados')

# Gráfica 2: Ingresos por Clúster
plt.subplot(1, 2, 2)
sns.boxplot(x='Cluster', y='Income', data=df, palette='viridis')
plt.title('Ingresos Anuales por Segmento')
plt.xlabel('Segmento (Clúster)')
plt.ylabel('Dólares (Ingreso)')

plt.tight_layout()
plt.show()

### PCA Optimizado

In [ ]:
# 1. Aplicar PCA a la matriz purgada y optimizada
pca_opt = PCA(n_components=2)
componentes_optimizados = pca_opt.fit_transform(df_core_scaled)

# 2. Crear tabla temporal para graficar
df_pca_opt = pd.DataFrame(data=componentes_optimizados, columns=['PC1', 'PC2'])
df_pca_opt['Cluster'] = df['Cluster'].values # Le pegamos las nuevas etiquetas

# 3. Generar el gráfico
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Cluster', 
    data=df_pca_opt, 
    palette='viridis', 
    s=70, 
    alpha=0.8
)

plt.title('Visualización Optimizada de Segmentos (Variables Financieras)')
plt.xlabel('Componente Principal 1 (Poder Adquisitivo)')
plt.ylabel('Componente Principal 2 (Hábitos de Compra)')
plt.legend(title='Segmento')
plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

Se puede observar que el clustering funciono perfectamente y se visualiza la distincion en los tres grupos.

## Conclusión Ejecutiva y Estrategia de Negocio

A partir de la optimización del modelo K-Means (k=3) y la reducción de dimensionalidad con Análisis de Componentes Principales (PCA), hemos logrado aislar el ruido generado por variables categóricas atípicas. Al enfocar el algoritmo en métricas puramente financieras, segmentamos la base de clientes en tres perfiles de alto valor interpretativo.

A continuación se detallan los diagnósticos y las acciones comerciales recomendadas para maximizar el Retorno de Inversión (ROI):

### Clúster 2: Clientes Premium (Alto Valor)
* **Perfil Demográfico y Financiero:** Poseen los ingresos promedios más altos de la base (~$73,800) y, en consecuencia, el mayor Gasto Total (~$1,298). Su edad promedio es de 58 años.
* **Diagnóstico:** Son el motor principal de ingresos de la compañía. Presentan una nula o muy baja sensibilidad a los descuentos.
* **Estrategia Recomendada (Retención mediante Estatus):** 
  * Excluir a este segmento de las campañas masivas de rebajas.
  * Diseñar programas de lealtad basados en niveles VIP (ej. acceso anticipado a nuevas colecciones).
  * Ofrecer atención personalizada, servicios de *concierge* o productos exclusivos de alto margen (como vinos de reserva o metales preciosos).

### Clúster 0: Clientes Conservadores (Valor Medio)
* **Perfil Demográfico y Financiero:** Muestran ingresos estables (~$49,900) y un gasto moderado (~$445). Es el grupo demográfico de mayor edad (63 años).
* **Diagnóstico:** Son clientes tradicionales y constantes que proporcionan estabilidad al volumen de ventas base.
* **Estrategia Recomendada (Cross-Selling y Fidelización):**
  * Implementar campañas de venta cruzada (*cross-selling*) sugiriendo productos que complementen sus categorías de compra históricas.

### Clúster 1: Cazadores de Ofertas (Sensibles al Precio)
* **Perfil Demográfico y Financiero:** Es el grupo mayoritario en volumen de personas, pero registran los ingresos más bajos (~$33,400) y el gasto mínimo (~$128). Su edad promedio es de 50 años.
* **Diagnóstico:** Existe una alta fricción para que adquieran productos a precio regular; su principal detonante de conversión es el ahorro percibido.
* **Estrategia Recomendada (Rotación de Inventario):**
  * Dirigir a este segmento el grueso de las campañas de liquidación (*Clearance*).
  * Utilizar cupones de descuento agresivos (ej. 2x1, 50% off) para incentivar el volumen y la frecuencia de compra.
  